# Pandas to SFS Conversion Example

This notebook shows how to convert lidar and camera data into Scale's SFS format.

You can adjust the steps to fit your own data, which hopefully won't be too different.

We're using this example PandaSet dataset [here](https://pandaset.org/).

In [1]:
from dataclasses import asdict
import glob
import json
import os
import pprint
import sys
from typing import Dict, List, Optional, Union

import numpy as np
import pandas as pd

#  !{sys.executable} -m pip install scale_sensor_fusion_io scale_json_binary

import scale_sensor_fusion_io as sfio
from scale_sensor_fusion_io.model_converters.sfs import to_scene_spec_sfs
from scale_sensor_fusion_io.utils.video_helpers import generate_video
from scale_sensor_fusion_io.validation import parse_and_validate_scene
from scale_json_binary.json_binary_codecs import read_file

In [2]:
USEC_IN_SEC = int(1e6)

# choosing a sequence with annotations for the example
PANDASET_SEQ002_PATH = os.path.expanduser('~/pandaset_0/002/')
PANDASET_LIDAR_PATH = os.path.join(PANDASET_SEQ002_PATH, 'lidar')
PANDASET_CAMERA_PATH = os.path.join(PANDASET_SEQ002_PATH, 'camera')

In [3]:
camera_names = next(os.walk(PANDASET_CAMERA_PATH))[1]
camera_names

['front_camera',
 'left_camera',
 'back_camera',
 'right_camera',
 'front_left_camera',
 'front_right_camera']

## Adjust Timestamps

PandaSet stores per-frame timestamps in their own file for lidar and each camera. Per-point timestamps for the lidar are stored with the rest of the point cloud data in Pandas DataFrames, one DataFrame per lidar frame.

**IMPORTANT:** We first need to adjust timestamps in the following ways:

1. Use microseconds
2. First timestamp is 0
3. Each lidar frame timestamp is less than or equal to every lidar point timestamp in the frame
4. Timestamps for each frame should be monotonically increasing (make sure your frames are in order)

We satisfy point 2 by subtracting the minimum timestamp from all timestamps in our data.

In [4]:
### LOAD TIMESTAMPS ###

with open(os.path.join(PANDASET_LIDAR_PATH, "timestamps.json"), "r") as f:
    lidar_timestamps = json.load(f)  # frame-level timestamps

# each point cloud comes from a separate file named 00.pkl.gz, 01.pkl.gz, 02.pkl.gz, ...
lidar_dataframes = [
    pd.read_pickle(fp)
    for fp in sorted(glob.glob(os.path.join(PANDASET_LIDAR_PATH, "*.pkl.gz")))
]

# adjust lidar frame timestamps to be less-than-or-equal-to the point timestamps within the frame
lidar_timestamps = [
    min(frame_timestamp, df["t"].min())
    for frame_timestamp, df in zip(lidar_timestamps, lidar_dataframes)
]

camera_timestamps = {}
for camera_name in camera_names:
    with open(
        os.path.join(PANDASET_CAMERA_PATH, camera_name, "timestamps.json"), "r"
    ) as f:
        camera_timestamps[camera_name] = json.load(f)


In [5]:
### CONVERT TO MICROSECONDS AND ADJUST SO FIRST TIMESTAMP IS 0 ###

minimum_timestamp = int(
    min(
        [lidar_timestamps[0]]
        + [timestamps[0] for timestamps in camera_timestamps.values()]
    )
    * USEC_IN_SEC
)

lidar_timestamps = [
    int(timestamp * USEC_IN_SEC) - minimum_timestamp for timestamp in lidar_timestamps
]

for df in lidar_dataframes:
    df["t"] = df["t"].apply(lambda t: int(t * USEC_IN_SEC) - minimum_timestamp)

for camera_name in camera_names:
    camera_timestamps[camera_name] = [
        int(timestamp * USEC_IN_SEC) - minimum_timestamp
        for timestamp in camera_timestamps[camera_name]
    ]

In [6]:
### VALIDATE TIMESTAMPS ###

assert len(lidar_timestamps) == len(lidar_dataframes), (
    f"Lengths do not match: len(lidar_dataframes)={len(lidar_dataframes)} "
    "len(lidar_timestamps)={len(lidar_timestamps)}"
)

for i in range(1, len(lidar_timestamps)):
    assert lidar_timestamps[i - 1] < lidar_timestamps[i],\
        "Lidar timestamps are not monotonically increasing"

for camera_name, timestamps in camera_timestamps.items():
    num_images = len(
        glob.glob(os.path.join(PANDASET_CAMERA_PATH, camera_name, f"*.jpg"))
    )
    assert num_images == len(camera_timestamps[camera_name]), (
        f"Lengths do not match: len({camera_name}_images)={num_images} "
        "len({camera_name}_timestamps)={len(camera_timestamps[camera_name])}"
    )

    for i in range(1, len(timestamps)):
        assert timestamps[i - 1] < timestamps[i],\
            f"{camera_name} timestamps are not monotonically increasing"

## LIDAR

PandaSet stores lidar point clouds as Pandas DataFrames with each DataFrame corresponding to a frame captured by the lidar. The lidar's pose is stored in its own file as an array with each element corresponding to a frame.

In [7]:
def generate_lidar_sensor(
    dataframes: List[pd.DataFrame], timestamps: List[int]
) -> sfio.LidarSensor:
    ### CREATE LIDAR FRAMES ###

    lidar_points = [
        sfio.LidarSensorPoints(
            positions=df[["x", "y", "z"]].to_numpy(dtype=np.float32),
            timestamps=df["t"].to_numpy(dtype=np.uint32),
            intensities=df["i"].to_numpy(dtype=np.uint8),
        )
        for df in dataframes
    ]
    frames = [
        sfio.LidarSensorFrame(points=points, timestamp=timestamp)
        for points, timestamp in zip(lidar_points, timestamps)
    ]

    ### LIDAR POSES ###

    with open(os.path.join(PANDASET_LIDAR_PATH, "poses.json"), "r") as f:
        poses_json = json.load(f)  # array of poses for each frame
    # PosePath expects an array of poses with shape (N, 7), where N is the number of poses
    poses = sfio.PosePath(
        data=np.fromiter(
            (
                (
                    pose["position"]["x"],
                    pose["position"]["y"],
                    pose["position"]["z"],
                    pose["heading"]["x"],
                    pose["heading"]["y"],
                    pose["heading"]["z"],
                    pose["heading"]["w"],
                )
                for pose in poses_json
            ),
            dtype=np.dtype((float, 7)),
            count=len(poses_json),
        ),
        index=timestamps,  # index is required
    )
    assert len(poses) == len(
        frames
    ), f"Lengths do not match: len(poses)={len(poses)} len(frames)={len(frames)}"

    # this dataset has points on a world coordinate system
    return sfio.LidarSensor(
        id="lidar_0", poses=poses, frames=frames, coordinates="world"
    )

## Camera

This dataset has multiple cameras, so we'll create a `CameraSensor` for each camera. Similar to the lidar data, each camera's pose is stored as an array in its own file with each element corresponding to a frame.

**NOTE:** It is important to use the `scale_sensor_fusion_io.utils.generate_video` utility to create a video from images as it uses a specific encoding. Videos are also better for compression.

In [8]:
CAMERA_WIDTH = 1920
CAMERA_HEIGHT = 1080
CAMERA_FPS = 10


def generate_camera_sensor(camera_name: str, timestamps: List[int]) -> sfio.CameraSensor:
    ### CAMERA INTRINSICS ###

    with open(
        os.path.join(PANDASET_CAMERA_PATH, camera_name, "intrinsics.json"), "r"
    ) as f:
        intrinsics_json = json.load(f)
    intrinsics = sfio.CameraIntrinsics(
        fx=intrinsics_json["fx"],
        fy=intrinsics_json["fy"],
        cx=intrinsics_json["cx"],
        cy=intrinsics_json["cy"],
        width=CAMERA_WIDTH,
        height=CAMERA_HEIGHT,
        distortion=None,
    )

    ### CAMERA POSES ###

    with open(os.path.join(PANDASET_CAMERA_PATH, camera_name, "poses.json"), "r") as f:
        poses_json = json.load(f)  # array of poses for each frame
    # PosePath expects an array of poses with shape (N, 7), where N is the number of poses
    poses = sfio.PosePath(
        data=np.fromiter(
            (
                (
                    pose["position"]["x"],
                    pose["position"]["y"],
                    pose["position"]["z"],
                    pose["heading"]["x"],
                    pose["heading"]["y"],
                    pose["heading"]["z"],
                    pose["heading"]["w"],
                )
                for pose in poses_json
            ),
            dtype=np.dtype((float, 7)),
            count=len(poses_json),
        ),
        index=timestamps,  # index is required
    )
    assert len(poses) == len(timestamps), f"Lengths do not match: len(poses)={len(poses)} len(timestamps)={len(timestamps)}"

    ### GENERATE VIDEO ###

    # use this utility function to encode the video correctly
    generate_video(
        # PandaSet images are stored as 00.jpg, 01.jpg, 02.jpg, ...
        image_files=sorted(
            glob.glob(os.path.join(PANDASET_CAMERA_PATH, camera_name, f"*.jpg"))
        ),
        target_file=os.path.join(PANDASET_CAMERA_PATH, camera_name, "video.mp4"),
        fps=CAMERA_FPS,
    )
    video = sfio.CameraSensorVideo(
        timestamps=timestamps,
        content=np.fromfile(
            os.path.join(PANDASET_CAMERA_PATH, camera_name, "video.mp4"), dtype=np.uint8
        ),
        fps=CAMERA_FPS,
    )

    return sfio.CameraSensor(
        id=camera_name,
        intrinsics=intrinsics,
        video=video,
        poses=poses,
    )

## Annotations

This dataset has some cuboid annotations for the lidar already. They are stored in Pandas DataFrames with one DataFrame per lidar frame. Cuboids can be joined across frames using their uuid.

In [9]:
ATTRIBUTES = [
    "attributes.object_motion",
    "attributes.pedestrian_behavior",
    "attributes.pedestrian_age",
]


class AttributeParseData:
    """For collecting attributes in a CuboidAnnotationParseData of a single attribute

    Members:
        timestamps (List[int]): timestamps for the values
        values (List[str]): list of values over time
    """

    timestamps: List[int]
    values: List[Union[str, int, List[str]]]

    def __init__(self):
        self.timestamps = []
        self.values = []


class CuboidAnnotationParseData:
    """For collecting data referring to a single uuid

    Members:
        attribute_map (Dict[str, AttributeParseData]): attributes found for this CuboidAnnotationParseData
        label (Optional[str]): label of this annotation
        spatial_data (List[List[float]]): [[x, y, z, yaw, pitch, roll, dx, dy, dz], ...]
            spatial information over time
        spatial_timestamps (List[int]): timestamps of spatial_data
        stationary (Optional[bool]): if this annotation is stationary
    """

    attribute_map: Dict[str, AttributeParseData]
    label: Optional[str]
    spatial_data: List[List[float]]
    spatial_timestamps: List[int]
    stationary: Optional[bool]

    def __init__(self):
        self.attribute_map = {}
        self.label = None
        self.spatial_data = []
        self.spatial_timestamps = []
        self.stationary = None

    def add_attribute(self, attribute: str, timestamp: int, value: str):
        """Adds to attribute_map"""

        if attribute not in self.attribute_map:
            self.attribute_map[attribute] = AttributeParseData()
        self.attribute_map[attribute].timestamps.append(timestamp)
        self.attribute_map[attribute].values.append(value)


def generate_cuboids(lidar_timestamps: List[int]) -> List[sfio.CuboidAnnotation]:
    # each frame has a separate file named 00.pkl.gz, 01.pkl.gz, 02.pkl.gz, ...
    cuboid_dataframes = [
        pd.read_pickle(fp)
        for fp in sorted(
            glob.glob(
                os.path.join(PANDASET_SEQ002_PATH, "annotations", "cuboids", "*.pkl.gz")
            )
        )
    ]

    ### AGGREGATE EACH CUBOID ACROSS FRAMES ###

    annotation_map: Dict[
        str, CuboidAnnotationParseData
    ] = {}  # uuid: CuboidAnnotationParseData
    for df, timestamp in zip(cuboid_dataframes, lidar_timestamps):
        for _, row in df.iterrows():
            if row["uuid"] not in annotation_map:
                annotation_map[row["uuid"]] = CuboidAnnotationParseData()
            annotation = annotation_map[row["uuid"]]
            annotation.spatial_data.append(
                [
                    row["position.x"],
                    row["position.y"],
                    row["position.z"],
                    row["yaw"],
                    0,  # pitch
                    0,  # roll
                    row["dimensions.x"],
                    row["dimensions.y"],
                    row["dimensions.z"],
                ]
            )
            annotation.spatial_timestamps.append(timestamp)
            if pd.notnull(row["label"]):
                annotation.label = row["label"]
            if pd.notnull(row["stationary"]):
                annotation.stationary = row["stationary"]
            for attribute in ATTRIBUTES:
                if pd.notnull(row[attribute]):
                    annotation.add_attribute(attribute, timestamp, row[attribute])
    cuboids: List[sfio.CuboidAnnotation] = []
    for uuid, annotation in annotation_map.items():
        # CuboidPath expects an array of cuboids with shape (N, 9)
        # this represents a single cuboid appearing in N frames
        cuboid_path = sfio.CuboidPath(
            data=np.array(
                annotation.spatial_data,
                dtype=float,
            ),
            index=annotation.spatial_timestamps,  # index is required
        )
        attributes = [
            sfio.AttributePath(
                name=attribute_name,
                timestamps=attribute.timestamps,
                values=attribute.values,
            )
            for attribute_name, attribute in annotation.attribute_map.items()
        ]
        cuboids.append(
            sfio.CuboidAnnotation(
                id=uuid,
                path=cuboid_path,
                label=annotation.label,
                stationary=annotation.stationary,
                attributes=attributes,
            )
        )
    return cuboids

## Create SFS Scene

In [10]:
lidar_sensor = generate_lidar_sensor(lidar_dataframes, lidar_timestamps)
camera_sensors = [
    generate_camera_sensor(camera_name, camera_timestamps[camera_name])
    for camera_name in camera_names
]
sensors = camera_sensors + [lidar_sensor]
annotations = generate_cuboids(lidar_timestamps)

scene = sfio.Scene(
    sensors=sensors, annotations=annotations, time_offset=minimum_timestamp, time_unit="microseconds"  # type: ignore
)
# convert the Scene object to an sfs object
sfs_scene = to_scene_spec_sfs(scene)

100%|██████████| 80/80 [00:00<00:00, 83.90it/s] 


In [11]:
encoder = sfio.JSONBinaryEncoder()
encoder.write_file(os.path.join(PANDASET_SEQ002_PATH, "scene.sfs"), sfs_scene) # type: ignore

## Test

The following code will validate an SFS file. You can also use LidarLite to view the scene online by dragging and dropping the file: https://dashboard.scale.com/lidarlite/.

In [12]:
pp = pprint.PrettyPrinter(depth=6)


def test0(sfs_scene_path):
    print("Test 0")
    raw_data = read_file(sfs_scene_path)

    result = parse_and_validate_scene(raw_data)

    if not result.success:
        pp.pprint(asdict(result))
    else:
        print("Scene parsed successfully")

In [13]:
test0(os.path.join(PANDASET_SEQ002_PATH, "scene.sfs"))

Test 0
Scene parsed successfully
